# Hanoi (Python)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fabiofurini/algorithms/blob/main/notebooks/ch14_recursion/Hanoi_python.ipynb)

Chapter 14 — Recursion. Runnable notebook of `Hanoi`: the algorithm in Python, run on the examples of the chapter, with the «Try it» cell to run it on your own data. The pseudo-code, the examples and the proofs are on the [chapter page](https://fabiofurini.github.io/algorithms/recursion/14-recursion/).

The same notebook in [C++](https://colab.research.google.com/github/fabiofurini/algorithms/blob/main/notebooks/ch14_recursion/Hanoi_cpp.ipynb) and [Java](https://colab.research.google.com/github/fabiofurini/algorithms/blob/main/notebooks/ch14_recursion/Hanoi_java.ipynb); all the algorithms of the chapter in the [chapter notebook](https://colab.research.google.com/github/fabiofurini/algorithms/blob/main/notebooks/ch14_recursion_python.ipynb).

**How to run it.** Click the «Open in Colab» badge, then *Runtime → Run all* (Ctrl+F9), or run one cell at a time with Shift+Enter, from top to bottom. Python 3, nothing to install.

**Names.** There are no arrays in this chapter: variables and parameters have the names of the pseudo-code. Two exceptions in Python, where `from` and `next` have a special meaning: `Hanoi` uses `frm` instead of `from` and `IterativeFib` uses `nxt` instead of `next`.

## Tools: counters, stack and trace

The cell defines the counters (`cnt`, `calls`, `max_depth`), the call stack (`push` and `pop`) and the functions that print the trace (`event`, `say`) and the table of steps (`print_steps`). It has to be run only once.

In [1]:
cnt = [0] * 20      # cnt[r] = number of executions of line r (lines numbered as in the lecture notes)
calls = 0           # number of calls = nodes of the tree of recursive calls
max_depth = 0       # depth of the recursion = maximum height of the stack
mults = 0           # multiplications (FastPower)
moves = 0           # moves (Hanoi)
ev = 0              # counter of the events of the trace
stack = []          # the call stack: the arguments of the active calls
seen = [0] * 64     # seen[x] = number of calls with argument x
trace = False       # True: print the trace of the execution


def reset():
    global calls, max_depth, mults, moves, ev
    calls = max_depth = mults = moves = ev = 0
    stack.clear()
    for r in range(len(cnt)):
        cnt[r] = 0
    for k in range(len(seen)):
        seen[k] = 0


def push(x):                       # a call starts: its frame goes on top of the stack
    global calls, max_depth
    calls += 1
    stack.append(x)
    max_depth = max(max_depth, len(stack))
    if x < len(seen):
        seen[x] += 1


def pop():                         # the call returns: its frame leaves the stack
    stack.pop()


def event(text):                   # numbered line of the trace, with the stack
    global ev
    ev += 1
    print("%2d. %s   stack: %s" % (ev, text, stack))


def say(call, action):             # line of the trace: the call, indented one level for every active call
    print("%-28s| %s" % ("  " * (len(stack) - 1) + call, action))


def print_steps(*lines):
    print("Line | Executions")
    for r in lines:
        print("%4d | %10d" % (r, cnt[r]))
    print("Total steps S = %d" % sum(cnt))

## Towers of Hanoi: Hanoi

**Input:** the number of disks $n \ge 1$, the source peg $from$, the destination peg $to$ and the auxiliary peg $aux$. **Output:** the sequence of moves that transfers disks $1, \dots, n$ from $from$ to $to$. $M(n) = 2^n - 1$ moves (optimal), $S(n) = 3 \cdot 2^n - 4 = \Theta(2^n)$ steps, depth $n$.

Pseudo-code: section 14.8 of the [chapter page](https://fabiofurini.github.io/algorithms/recursion/14-recursion/).

In [2]:
def move(disk, frm, to):                           # "move disk d from from to to": counts the move and, if show_moves, prints it
    global moves
    moves += 1
    if show_moves:
        print("%d. move disk %d from %s to %s" % (moves, disk, frm, to))


show_moves = False


def hanoi(n, frm, to, aux):                        # in Python "from" is a reserved word: frm
    push(n); cnt[2] += 1
    call = "Hanoi(%d,%s,%s,%s)" % (n, frm, to, aux) if trace else ""
    if n == 1:
        cnt[3] += 1
        move(1, frm, to)
        if trace: say(call, "prints: move disk 1 from %s to %s, returns" % (frm, to))
    else:
        if trace: say(call, "calls Hanoi(%d,%s,%s,%s)" % (n - 1, frm, aux, to))
        cnt[5] += 1
        hanoi(n - 1, frm, aux, to)
        cnt[6] += 1
        move(n, frm, to)
        if trace: say(call, "prints: move disk %d from %s to %s, calls Hanoi(%d,%s,%s,%s)" % (n, frm, to, n - 1, aux, to, frm))
        cnt[7] += 1
        hanoi(n - 1, aux, to, frm)
        if trace: say(call, "returns")
    pop()

In [3]:
# Example of the lecture notes: Hanoi(3, A, C, B), the moves
reset()
show_moves = True
hanoi(3, "A", "C", "B")
show_moves = False
print("%d moves, %d calls, depth of the recursion = %d" % (moves, calls, max_depth))

# The same execution, call by call (indentation = depth in the stack)
print()
reset()
trace = True
print("%-28s| %s" % ("call", "action"))
hanoi(3, "A", "C", "B")
trace = False
print_steps(2, 3, 5, 6, 7)
print("S(n) = 3 * 2^n - 4 = %d" % (3 * 2 ** 3 - 4))

# Moves and steps: M(n) = 2^n - 1 and S(n) = 3 * 2^n - 4
print()
print("  n |  moves | 2^n - 1 |    calls |   S(n) | 3 * 2^n - 4 | depth")
for n in range(1, 11):
    reset()
    hanoi(n, "A", "C", "B")
    print("%3d | %6d | %7d | %8d | %6d | %11d | %d" % (n, moves, 2 ** n - 1, calls, sum(cnt), 3 * 2 ** n - 4, max_depth))

# The legend: 64 disks, one move per second
print()
print("n = 64: 2^64 - 1 seconds are more than %.1e years" % ((2.0 ** 64 - 1) / (365.25 * 24 * 3600)))

1. move disk 1 from A to C
2. move disk 2 from A to B
3. move disk 1 from C to B
4. move disk 3 from A to C
5. move disk 1 from B to A
6. move disk 2 from B to C
7. move disk 1 from A to C
7 moves, 7 calls, depth of the recursion = 3

call                        | action
Hanoi(3,A,C,B)              | calls Hanoi(2,A,B,C)
  Hanoi(2,A,B,C)            | calls Hanoi(1,A,C,B)
    Hanoi(1,A,C,B)          | prints: move disk 1 from A to C, returns
  Hanoi(2,A,B,C)            | prints: move disk 2 from A to B, calls Hanoi(1,C,B,A)
    Hanoi(1,C,B,A)          | prints: move disk 1 from C to B, returns
  Hanoi(2,A,B,C)            | returns
Hanoi(3,A,C,B)              | prints: move disk 3 from A to C, calls Hanoi(2,B,C,A)
  Hanoi(2,B,C,A)            | calls Hanoi(1,B,A,C)
    Hanoi(1,B,A,C)          | prints: move disk 1 from B to A, returns
  Hanoi(2,B,C,A)            | prints: move disk 2 from B to C, calls Hanoi(1,A,C,B)
    Hanoi(1,A,C,B)          | prints: move disk 1 from A to C, returns
 

**Try it yourself.** Change the number of disks and count the moves.

In [4]:
n = 4        # <-- change the number of disks
reset()
show_moves = True
hanoi(n, "A", "C", "B")
show_moves = False
print("%d moves = 2^%d - 1 = %d" % (moves, n, 2 ** n - 1))

1. move disk 1 from A to B
2. move disk 2 from A to C
3. move disk 1 from B to C
4. move disk 3 from A to B
5. move disk 1 from C to A
6. move disk 2 from C to B
7. move disk 1 from A to B
8. move disk 4 from A to C
9. move disk 1 from B to C
10. move disk 2 from B to A
11. move disk 1 from C to A
12. move disk 3 from B to C
13. move disk 1 from A to B
14. move disk 2 from A to C
15. move disk 1 from B to C
15 moves = 2^4 - 1 = 15


---

Teaching material by [Fabio Furini](https://fabiofurini.github.io/) — DIAG, Sapienza University of Rome.
Text, figures and data [CC BY 4.0](https://github.com/fabiofurini/algorithms/blob/main/LICENSE), code [MIT](https://github.com/fabiofurini/algorithms/blob/main/LICENSE-CODE).